# SmolVLM-256M: EVLM Benchmark

This notebook runs the benchmark of the EVLM paper (Farahani et al., *EVLM: Intent-Driven Edge Vision Language Model for UAV-Based Power Line Inspection*, EDGE 2026) with the fastest inference setup of this repository.

The paper uses two public datasets (Section V-B):

- **Power line videos** (Xing et al., IROS 2023, ref. [35]): 720x480 frames at 30 fps from eight environments. The paper cuts them into 4 s clips of 120 frames. This dataset is used for the speed tests.
- **InsPLAD** (Vieira e Silva et al., 2023, ref. [43]): UAV images of power line parts with labels. We use the fault classification crops (good, rust and other defects). This dataset is used for the accuracy test.

`get_data.py` downloads only the parts that the test needs: one 4 s clip for each environment (about 0.5 GB) and the InsPLAD fault crops (about 0.9 GB). The full archives are 25 GB and 6.4 GB.

`evlm_eval.py` has two tasks:

1. `clips`: for each clip, a histogram filter keeps the key frames (512 bins, threshold 0.95, as in the paper). The VLM answers the operator intent for each key frame. The script prints the key frames, the filter time, the VLM time and the segment latency.
2. `insplad`: the VLM answers the two prompts of the paper's Table III for each image. The script scores the component name and the rust answer against the labels.

The defaults are the best setup from the earlier tests: fast GPU preprocessing, `--size 1024`, CUDA-graph decoding, stop at EOS, and a TensorRT engine for SigLIP. Each part has a flag, so the last cell can turn the parts off one at a time (ablation).

We use the base model. The paper fine-tunes it with LoRA, and its weights are not public, so the accuracy here is lower than in the paper.

In [2]:
REPO = "./edge-ai-smol-lvm-250-optimization"
DATA = f"{REPO}/data"
OUT = f"{REPO}/results"

# Colab: packages for the TensorRT engine. The Jetson has TensorRT in JetPack.
!pip install -q tensorrt onnx

# dataset setup: about 1.4 GB, only the files the test needs; files that exist already are skipped
!python {REPO}/get_data.py --out {DATA}

python3: can't open file '/content/./edge-ai-smol-lvm-250-optimization/get_data.py': [Errno 2] No such file or directory


In [ ]:
# speed test on the power line clips, best setup
!python {REPO}/evlm_eval.py --task clips --data {DATA} --out {OUT}

In [ ]:
# accuracy test on InsPLAD (10 images for each asset and label), best setup
!python {REPO}/evlm_eval.py --task insplad --data {DATA} --out {OUT} --limit 10

In [ ]:
# ablation: turn off one part at a time (two environments, to keep the run short)
import json
import subprocess

ENVS = "forest,sand"
CONFIGS = {
    "best": [],
    "no TensorRT": ["--no-trt"],
    "default tiles": ["--size", "0"],
    "no stop at EOS": ["--no-stop"],
    "eager decode": ["--decode", "eager"],
    "bare generate": ["--decode", "bare"],
    "slow preprocess": ["--preprocess", "slow"],
    "no frame filter": ["--filter", "none"],
    "nothing (baseline)": ["--preprocess", "slow", "--size", "0", "--decode", "bare", "--no-trt"],
}

print(f"{'config':<20} {'key frames':>10} {'VLM ms/call':>11} {'tokens':>6} {'ms/token':>8} {'segment s':>9}")
for name, flags in CONFIGS.items():
    r = subprocess.run(
        ["python", f"{REPO}/evlm_eval.py", "--task", "clips", "--data", DATA, "--out", OUT, "--envs", ENVS, *flags],
        capture_output=True, text=True,
    )
    line = next((l for l in r.stdout.splitlines() if l.startswith("SUMMARY")), None)
    if line is None:
        print(f"{name:<20} failed: {(r.stderr.strip().splitlines() or ['?'])[-1]}")
        continue
    s = json.loads(line[len("SUMMARY "):])
    print(f"{name:<20} {s['key_frames']:10.2f} {s['vlm_ms_per_call']:11.1f} {s['tokens_per_answer']:6.1f} "
          f"{s['decode_ms_per_token']:8.2f} {s['segment_s']:9.2f}")

## Comparison with the paper (run on the Jetson Orin)

The paper ran on an NVIDIA Jetson (8-core CPU, 16 GB, 1024 CUDA cores, 32 Tensor cores). Run the next cell on that device. Times from another GPU cannot be compared with the paper.

The cell runs two setups on all 8 environments:

- `paper-like baseline`: the plain model path (slow preprocessing, default tiles, Hugging Face `generate`, no TensorRT). This is our closest copy of the paper's pipeline. The paper does not give its inference code.
- `best`: all the speed-ups.

Compare the time for each VLM call, not the time for each segment. The paper's filter keeps about 3.45 frames per segment, and our clips can keep a different number. The last column scales our segment to 3.45 key frames.

The paper's time for each call is an estimate: 6.448 s per segment / 3.45 calls = about 1.87 s.

Also compare the tokens per answer, because a longer answer takes more time. The paper uses a fine-tuned model, so its answers can be longer or shorter.

The segment time is the filter time plus the VLM calls. The PNG decode is not included, because the paper decodes video and we decode PNG files.

In [ ]:
# Orin comparison: run on the Jetson Orin, all 8 environments
import json
import subprocess

PAPER_SEGMENT_S, PAPER_KEY_FRAMES = 6.448, 3.45  # Table V (EVLM) and the FOT frames per segment
SETUPS = {
    "paper-like baseline": ["--preprocess", "slow", "--size", "0", "--decode", "bare", "--no-trt"],
    "best": [],
}

results = {}
for name, flags in SETUPS.items():
    r = subprocess.run(
        ["python", f"{REPO}/evlm_eval.py", "--task", "clips", "--data", DATA, "--out", OUT, *flags],
        capture_output=True, text=True,
    )
    line = next((l for l in r.stdout.splitlines() if l.startswith("SUMMARY")), None)
    if line is None:
        print(f"{name}: failed: {(r.stderr.strip().splitlines() or ['?'])[-1]}")
        continue
    results[name] = json.loads(line[len("SUMMARY "):])

print(f"{'setup':<20} {'key frames':>10} {'VLM s/call':>10} {'prefill ms':>10} {'tokens':>6} {'ms/token':>8} "
      f"{'segment s':>9} {'segment s at 3.45 keys':>22}")
for name, s in results.items():
    at_paper = s["filter_s_per_segment"] + PAPER_KEY_FRAMES * s["vlm_ms_per_call"] / 1e3
    print(f"{name:<20} {s['key_frames']:10.2f} {s['vlm_ms_per_call'] / 1e3:10.2f} {s['prefill_ms']:10.1f} "
          f"{s['tokens_per_answer']:6.1f} {s['decode_ms_per_token']:8.2f} {s['segment_s']:9.2f} {at_paper:22.2f}")
print(f"{'paper (EVLM)':<20} {PAPER_KEY_FRAMES:10.2f} {PAPER_SEGMENT_S / PAPER_KEY_FRAMES:10.2f} {'-':>10} {'-':>6} "
      f"{'-':>8} {PAPER_SEGMENT_S:9.2f} {PAPER_SEGMENT_S:22.2f}")

if len(results) == 2:
    base, best = results["paper-like baseline"], results["best"]
    print(f"speed-up for each VLM call, best against baseline: {base['vlm_ms_per_call'] / best['vlm_ms_per_call']:.2f}x")